# 02 · Data Quality Assessment & Cleaning

**Purpose:** demonstrate every data-quality issue in the raw extract with concrete example rows (before → after),
run the cleaning pipeline, and reconcile what was *detected and repaired* against what the generator *injected*
(`data/raw/generation_manifest.json`).

The cleaning rules live in `src/data_cleaning.py` (rule IDs **DQ01-DQ20**); this notebook does not re-implement
them - it calls the pipeline and inspects its inputs and outputs. Spec §7 requirements covered:

| Spec §7 requirement | Rules |
|---|---|
| Duplicate appointment and work-order IDs | DQ01, DQ02 |
| Missing foreign keys / required fields | DQ06, DQ07, DQ09, DQ10, DQ12, DQ14, DQ17, DQ18 |
| Completion not earlier than start | DQ08 |
| Costs and revenue non-negative | DQ11, DQ13 |
| Ratings within 1-5 | DQ14 |
| Date formats and ranges | DQ04, DQ05 |
| Foreign-key relationships | DQ06, DQ10, DQ17, DQ18 |
| Impossible mileage / model-year | DQ15, DQ16 |
| Categorical consistency | DQ03, DQ20 |
| Reconciliation | DQ19 |

In [1]:
# Locate the project root (folder containing src/ and data/) so the notebook runs from anywhere.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(p for p in [_here, *_here.parents] if (p / "src").is_dir() and (p / "data").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: C:\Vehicle_Service_Operations_Analytics


In [2]:
import json
import warnings

import numpy as np
import pandas as pd

from src import config
from src.data_cleaning import CANONICAL_REGIONS, MODEL_ALIASES, load_raw, run

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

raw = load_raw()
manifest = json.loads((config.RAW_DIR / "generation_manifest.json").read_text(encoding="utf-8"))["injected_defects"]

# Run the full cleaning pipeline in memory (python -m src.data_cleaning writes the same outputs to disk).
clean = run(write=False)
print({k: len(v) for k, v in clean.items()})

{'service_centers': 8, 'technicians': 18, 'parts': 39, 'customers': 3351, 'vehicles': 5300, 'appointments': 26769, 'work_orders': 24092, 'part_usage': 46190, 'financials': 24092, 'feedback': 12186, 'fact_service': 24092, 'fact_appointments': 26769, 'fact_technician_month': 378, 'dim_date': 638}


### Consistency with the files on disk
The in-memory run must reproduce `data/cleaned/*.csv` exactly (row counts and keys), otherwise downstream SQL,
dashboard and analysis would be built on a stale extract.

In [3]:
chk = []
for name, df in clean.items():
    disk = pd.read_csv(config.CLEAN_DIR / f"{name}.csv", low_memory=False)
    key = df.columns[0]
    chk.append({"output": name, "rows_in_memory": len(df), "rows_on_disk": len(disk),
                "same_keys": set(df[key].astype(str)) == set(disk[key].astype(str))})
pd.DataFrame(chk)

,output,rows_in_memory,rows_on_disk,same_keys
0,service_centers,8,8,True
1,technicians,18,18,True
2,parts,39,39,True
3,customers,3351,3351,True
4,vehicles,5300,5300,True
5,appointments,26769,26769,True
6,work_orders,24092,24092,True
7,part_usage,46190,46190,True
8,financials,24092,24092,True
9,feedback,12186,12186,True


## 1. Duplicates (DQ01 exact rows, DQ02 conflicting keys)

Exact duplicate rows are typical of double-submitted forms or re-run exports. Example from `appointments`:

In [4]:
a_raw = raw["appointments"]
dups = a_raw[a_raw.duplicated(keep=False)].sort_values("appointment_id")
example_id = dups["appointment_id"].iloc[0]
print("BEFORE (raw):")
display(a_raw[a_raw["appointment_id"] == example_id])
print("AFTER (cleaned):")
display(clean["appointments"][clean["appointments"]["appointment_id"] == example_id])
pd.DataFrame({t: {"exact_duplicate_rows": int(raw[t].duplicated().sum())}
              for t in ["customers", "appointments", "work_orders", "financials", "feedback"]}).T

BEFORE (raw):


,appointment_id,vehicle_id,center_id,booking_date,scheduled_date,scheduled_slot,booking_channel,requested_service_type,status,cancellation_reason
5395,AP000165,V03604,C001,2025-01-01,2025-01-07,11:30,Ather App,Battery Health Check,No-Show,NaN
11456,AP000165,V03604,C001,2025-01-01,2025-01-07,11:30,Ather App,Battery Health Check,No-Show,NaN


AFTER (cleaned):


,appointment_id,vehicle_id,center_id,booking_date,scheduled_date,scheduled_slot,booking_channel,requested_service_type,status,cancellation_reason
5381,AP000165,V03604,C001,2025-01-01,2025-01-07,11:30,Ather App,Battery Health Check,No-Show,<NA>


,exact_duplicate_rows
customers,13
appointments,160
work_orders,96
financials,72
feedback,36


No *conflicting* duplicates (same key, different values) exist, so DQ02 removes nothing - it is kept as a guard.

## 2. Categorical inconsistencies (DQ03, DQ20)

Free-text entry produces case and whitespace variants that would fragment every GROUP BY.

In [5]:
cu_raw = raw["customers"].drop_duplicates()
bad_region = cu_raw[~cu_raw["region"].isin(CANONICAL_REGIONS)].head(4)
print("Customer region - BEFORE vs AFTER")
display(bad_region[["customer_id", "region"]].merge(clean["customers"][["customer_id", "region"]], on="customer_id",
                                                     suffixes=("_raw", "_clean")).assign(region_raw=lambda d: d["region_raw"].map(repr)))

v_raw = raw["vehicles"]
canon_models = set(MODEL_ALIASES.values())
bad_model = v_raw[~v_raw["model"].isin(canon_models)].head(4)
print("Vehicle model - BEFORE vs AFTER")
display(bad_model[["vehicle_id", "model"]].merge(clean["vehicles"][["vehicle_id", "model"]], on="vehicle_id",
                                                  suffixes=("_raw", "_clean")).assign(model_raw=lambda d: d["model_raw"].map(repr)))

bad_status = a_raw[~a_raw["status"].isin(["Completed", "Cancelled", "No-Show"])].drop_duplicates("appointment_id").head(5)
print("Appointment status - BEFORE vs AFTER")
display(bad_status[["appointment_id", "status"]].merge(clean["appointments"][["appointment_id", "status"]], on="appointment_id",
                                                        suffixes=("_raw", "_clean")).assign(status_raw=lambda d: d["status_raw"].map(repr)))

wo_raw = raw["work_orders"]
canon_st = set(clean["work_orders"]["service_type"].dropna())
bad_st = wo_raw[~wo_raw["service_type"].isin(canon_st)].drop_duplicates("work_order_id").head(4)
print("Service type - BEFORE vs AFTER")
display(bad_st[["work_order_id", "service_type"]].merge(clean["work_orders"][["work_order_id", "service_type"]], on="work_order_id",
                                                         suffixes=("_raw", "_clean")).assign(service_type_raw=lambda d: d["service_type_raw"].map(repr)))

Customer region - BEFORE vs AFTER


,customer_id,region_raw,region_clean
0,CU00932,'south',South
1,CU03346,'west',West
2,CU00880,'south',South
3,CU03212,'WEST',West


Vehicle model - BEFORE vs AFTER


,vehicle_id,model_raw,model_clean
0,V00053,'450 Apex',Ather 450 Apex
1,V00108,'450X',Ather 450X
2,V00201,'Ather 450x ',Ather 450X
3,V00214,'Ather 450x ',Ather 450X


Appointment status - BEFORE vs AFTER


,appointment_id,status_raw,status_clean
0,AP002739,' Completed',Completed
1,AP002178,'COMPLETED',Completed
2,AP010608,'completed',Completed
3,AP021356,' Completed',Completed
4,AP019244,'no-show',No-Show


Service type - BEFORE vs AFTER


,work_order_id,service_type_raw,service_type_clean
0,WO008960,'Periodic Service',Periodic Service
1,WO001433,'Periodic Service',Periodic Service
2,WO017568,'Belt Drive Service',Belt Drive Service
3,WO002728,'Periodic Service',Periodic Service


## 3. Date formats and ranges (DQ04, DQ05)

Some appointments were keyed in Indian `DD/MM/YYYY` format alongside ISO dates. Naïve parsing would either fail
or - worse - silently swap day and month for dates such as `05/03/2026`.

In [6]:
iso = pd.to_datetime(a_raw["scheduled_date"], format="ISO8601", errors="coerce")
ddmm = a_raw[iso.isna()].drop_duplicates("appointment_id").head(5)
display(ddmm[["appointment_id", "booking_date", "scheduled_date"]]
        .merge(clean["appointments"][["appointment_id", "scheduled_date"]], on="appointment_id", suffixes=("_raw", "_clean")))
ca = clean["appointments"]
print("Cleaned scheduled_date range:", ca["scheduled_date"].min(), "→", ca["scheduled_date"].max())
print("Bookings after scheduled date:", int((pd.to_datetime(ca["booking_date"]) > pd.to_datetime(ca["scheduled_date"])).sum()))

,appointment_id,booking_date,scheduled_date_raw,scheduled_date_clean
0,AP022295,2026-07-11,14/07/2026,2026-07-14
1,AP013227,2026-01-03,06/01/2026,2026-01-06
2,AP004000,2025-05-17,20/05/2025,2025-05-20
3,AP010019,2025-10-17,17/10/2025,2025-10-17
4,AP000640,2025-01-25,27/01/2025,2025-01-27


Cleaned scheduled_date range: 2025-01-01 → 2026-09-30
Bookings after scheduled date: 0


## 4. Impossible vehicle attributes (DQ15 model year, DQ16 mileage)

An Ather 450 Apex cannot have a 2022 model year (launched 2024), and a scooter cannot have negative mileage or
average more than 250 km/day since purchase.

In [7]:
first_year = v_raw["model"].str.strip().str.lower().map(MODEL_ALIASES).map(config.MODEL_FIRST_YEAR)
bad_year = v_raw[v_raw["model_year"] < first_year].head(5)
display(bad_year[["vehicle_id", "model", "purchase_date", "model_year"]]
        .merge(clean["vehicles"][["vehicle_id", "model_year"]], on="vehicle_id", suffixes=("_raw", "_clean")))
age = (pd.Timestamp(config.PERIOD_END) - pd.to_datetime(v_raw["purchase_date"])).dt.days.clip(lower=1)
bad_km = v_raw[(v_raw["mileage_km"] < 0) | (v_raw["mileage_km"] > age * config.MAX_PLAUSIBLE_KM_PER_DAY)].head(5)
display(bad_km[["vehicle_id", "purchase_date", "mileage_km"]]
        .merge(clean["vehicles"][["vehicle_id", "mileage_km"]], on="vehicle_id", suffixes=("_raw", "_clean")))

,vehicle_id,model,purchase_date,model_year_raw,model_year_clean
0,V00068,Ather 450X,2025-11-01,2019,2025
1,V00650,Ather 450X,2024-11-02,2018,2024
2,V00813,Ather Rizta,2026-05-22,2022,2026
3,V00871,Ather 450X,2021-08-11,2019,2021
4,V00906,Ather Rizta,2025-09-12,2021,2025


,vehicle_id,purchase_date,mileage_km_raw,mileage_km_clean
0,V01436,2024-06-07,475214,<NA>
1,V01490,2024-09-05,786322,<NA>
2,V01502,2025-07-03,856941,<NA>
3,V01916,2024-11-07,859494,<NA>
4,V02692,2025-01-30,857296,<NA>


Model year is imputed from the purchase year (the most defensible value); implausible mileage is set to null rather
than guessed, so mileage analyses simply exclude those vehicles.

## 5. Missing and orphan foreign keys (DQ06, DQ07, DQ10, DQ17, DQ18)

In [8]:
miss_c = a_raw[a_raw["center_id"].isna()].drop_duplicates("appointment_id").head(4)
print("DQ07 - missing centre imputed from the assigned technician's centre")
display(miss_c[["appointment_id", "center_id"]]
        .merge(wo_raw[["appointment_id", "technician_id"]].drop_duplicates("appointment_id"), on="appointment_id")
        .merge(clean["appointments"][["appointment_id", "center_id"]], on="appointment_id", suffixes=("_raw", "_clean")))

orphan_v = a_raw[~a_raw["vehicle_id"].isin(raw["vehicles"]["vehicle_id"])].drop_duplicates("appointment_id")
print(f"DQ06 - {len(orphan_v)} appointments reference vehicles that do not exist, e.g.:")
display(orphan_v.head(3)[["appointment_id", "vehicle_id", "status"]])
print("Still present after cleaning:", int(orphan_v["appointment_id"].isin(ca["appointment_id"]).sum()))

tech_ids = set(raw["technicians"]["technician_id"])
bad_tech = wo_raw[wo_raw["technician_id"].isna() | ~wo_raw["technician_id"].isin(tech_ids)].drop_duplicates("work_order_id")
print(f"DQ10 - {len(bad_tech)} work orders with a missing or unknown technician; values seen:",
      bad_tech["technician_id"].fillna("<null>").value_counts().to_dict())

DQ07 - missing centre imputed from the assigned technician's centre


,appointment_id,center_id_raw,technician_id,center_id_clean
0,AP026695,NaN,T001,C001
1,AP018466,NaN,T017,C008
2,AP022530,NaN,T015,C007
3,AP024629,NaN,T009,C004


DQ06 - 26 appointments reference vehicles that do not exist, e.g.:


,appointment_id,vehicle_id,status
2172,AP009561,V93220,Completed
2347,AP018075,V97510,Completed
3079,AP017000,V92759,Completed


Still present after cleaning: 0
DQ10 - 72 work orders with a missing or unknown technician; values seen: {'<null>': 48, 'T999': 24}


Work orders with an unknown technician are **kept** (the job happened and was billed) but excluded from technician
rankings and utilisation. Removing them would understate revenue.

## 6. Time-logic errors (DQ08 swapped times, DQ09 missing start)

In [9]:
w = wo_raw.drop_duplicates("work_order_id").copy()
st, et = pd.to_datetime(w["start_time"]), pd.to_datetime(w["end_time"])
swapped = w[et < st].head(4)
display(swapped[["work_order_id", "check_in_time", "start_time", "end_time"]]
        .merge(clean["work_orders"][["work_order_id", "start_time", "end_time"]], on="work_order_id", suffixes=("_raw", "_clean")))
ms = w[w["start_time"].isna()]
print(f"DQ09 - {len(ms)} distinct raw work orders have no start_time; "
      f"{int((~ms['appointment_id'].isin(ca['appointment_id'])).sum())} of them belonged to an orphan appointment removed under DQ06.")

,work_order_id,check_in_time,start_time_raw,end_time_raw,start_time_clean,end_time_clean
0,WO008631,2025-10-07 10:25:43,2025-10-07 13:47:27,2025-10-07 10:57:39,2025-10-07 10:57:39,2025-10-07 13:47:27
1,WO018529,2026-06-15 12:16:46,2026-06-15 14:15:12,2026-06-15 12:21:48,2026-06-15 12:21:48,2026-06-15 14:15:12
2,WO007031,2025-08-25 09:06:33,2025-08-25 10:46:13,2025-08-25 09:10:13,2025-08-25 09:10:13,2025-08-25 10:46:13
3,WO003401,2025-05-13 12:32:23,2025-05-14 10:40:22,2025-05-13 12:43:28,2025-05-13 12:43:28,2025-05-14 10:40:22


DQ09 - 24 distinct raw work orders have no start_time; 1 of them belonged to an orphan appointment removed under DQ06.


`end_time < start_time` always coincides with a plausible duration once the two are swapped, so the rule treats it
as a transposed entry rather than deleting the job. Rows with no start time keep their turnaround
(check-in → end) and financials; only wait time is unknown.

## 7. Financial errors (DQ11 negatives, DQ12 missing parts cost, DQ13 decimal shift)

In [10]:
f_raw = raw["financials"].drop_duplicates()
fc = clean["financials"]
neg = f_raw[(f_raw["revenue"] < 0) | (f_raw["labor_cost"] < 0)].head(4)
print("DQ11 - negative amounts (sign-entry errors)")
display(neg.merge(fc, on="work_order_id", suffixes=("_raw", "_clean"))[["work_order_id", "labor_cost_raw", "labor_cost_clean", "revenue_raw", "revenue_clean"]])

pu = clean["part_usage"].merge(clean["parts"][["part_id", "unit_cost"]], on="part_id")
pu_sum = (pu["quantity"] * pu["unit_cost"]).groupby(pu["work_order_id"]).sum().rename("part_usage_total")
mp = f_raw[f_raw["parts_cost"].isna()].head(4)
print("DQ12 - missing parts_cost recomputed from part_usage x unit_cost")
display(mp[["work_order_id", "parts_cost"]].merge(fc[["work_order_id", "parts_cost"]], on="work_order_id", suffixes=("_raw", "_clean"))
        .join(pu_sum, on="work_order_id"))

shift = f_raw[(f_raw["revenue"] > 10 * (f_raw["labor_cost"].abs() + f_raw["parts_cost"].fillna(0))) & (f_raw["revenue"] > 50_000)].head(4)
print("DQ13 - revenue keyed with the decimal point shifted (x100)")
display(shift[["work_order_id", "labor_cost", "parts_cost", "revenue"]].merge(fc[["work_order_id", "revenue"]], on="work_order_id", suffixes=("_raw", "_clean")))

DQ11 - negative amounts (sign-entry errors)


,work_order_id,labor_cost_raw,labor_cost_clean,revenue_raw,revenue_clean
0,WO019601,489.6,489.6,-1305.00,1305.00
1,WO012116,571.2,571.2,-2904.00,2904.00
2,WO001398,-265.5,265.5,562.50,562.50
3,WO018849,539.7,539.7,-3273.75,3273.75


DQ12 - missing parts_cost recomputed from part_usage x unit_cost


,work_order_id,parts_cost_raw,parts_cost_clean,part_usage_total
0,WO012327,NaN,0.0,NaN
1,WO000223,NaN,6100.0,6100.0
2,WO008557,NaN,250.0,250.0
3,WO023105,NaN,490.0,490.0


DQ13 - revenue keyed with the decimal point shifted (x100)


,work_order_id,labor_cost,parts_cost,revenue_raw,revenue_clean
0,WO016347,323.4,9800.0,1079000.0,10790.0
1,WO013409,900.0,1100.0,293000.0,2930.0
2,WO007007,527.1,430.0,118400.0,1184.0
3,WO009029,623.7,3890.0,556380.0,5563.8


The x100 rule is deliberately conservative: it requires revenue to exceed **10× the job's cost *and* ₹50,000**,
so a genuinely expensive motor-assembly job is not "corrected". All 24 injected shifts were caught with no false
positives (see reconciliation below).

## 8. Ratings and parts lines (DQ14, DQ17)

In [11]:
fb = raw["feedback"]
print("Raw rating values:", fb["rating"].value_counts(dropna=False).sort_index().to_dict())
print("Clean rating values:", clean["feedback"]["rating"].value_counts().sort_index().to_dict())
pur = raw["part_usage"]
display(pur[(pur["quantity"] <= 0) | (pur["part_id"] == "P999")].head(5))

Raw rating values: {0.0: 16, 1.0: 42, 2.0: 541, 3.0: 2531, 4.0: 5123, 5.0: 3993, 6.0: 14, 10.0: 19, nan: 24}
Clean rating values: {1: 42, 2: 540, 3: 2520, 4: 5108, 5: 3976}


,usage_id,work_order_id,part_id,quantity
301,PU000302,WO000161,P001,-1
337,PU000338,WO000180,P999,1
652,PU000653,WO000331,P001,0
736,PU000737,WO000374,P037,-1
1207,PU001208,WO000621,P037,0


Out-of-range and missing ratings are removed from the *feedback* sample only; the underlying work orders remain in
every operational and financial KPI.

## 9. Pipeline outputs: data-quality summary and cleaning log

In [12]:
summary = pd.read_csv(config.REPORTS_DIR / "data_quality_summary.csv")
log = pd.read_csv(config.REPORTS_DIR / "cleaning_log.csv")
display(summary)
log

,table,raw_rows,exact_duplicates,duplicate_keys,null_cells,columns_with_nulls,clean_rows,rows_removed,pct_retained
0,customers,3364,13,13,0,-,3351,13,99.61
1,vehicles,5300,0,0,0,-,5300,0,100.00
2,service_centers,8,0,0,0,-,8,0,100.00
3,technicians,18,0,0,0,-,18,0,100.00
4,appointments,26955,160,160,25178,"center_id (41), cancellation_reason (25137)",26769,186,99.31
5,work_orders,24213,96,96,72,"technician_id (48), start_time (24)",24092,121,99.50
6,parts,39,0,0,0,-,39,0,100.00
7,part_usage,46355,0,0,0,-,46190,165,99.64
8,financials,24189,72,72,36,parts_cost (36),24092,97,99.60
9,feedback,12303,36,36,24,rating (24),12186,117,99.05


,rule_id,table,column,issue,action,rows_affected
0,DQ01,customers,*,Exact duplicate rows,Removed,13
1,DQ03,customers,region,Case/whitespace variants,Standardised to canonical region,33
2,DQ02,customers,customer_id,Duplicate primary key with conflicting values,Kept first occurrence,0
3,DQ01,vehicles,*,Exact duplicate rows,Removed,0
4,DQ03,vehicles,model,Model name variants,Mapped to canonical model name,26
5,DQ15,vehicles,model_year,Model year before model launch or after purchase,Imputed from purchase_date year,21
6,DQ16,vehicles,mileage_km,Negative or > 250 km/day since purchase,Set to null (excluded from mileage analysis),15
7,DQ02,vehicles,vehicle_id,Duplicate primary key with conflicting values,Kept first occurrence,0
8,DQ01,appointments,*,Exact duplicate rows,Removed,160
9,DQ20,appointments,status,Status spelling/case variants,Mapped to Completed/Cancelled/No-Show,107


## 10. Reconciliation: injected (manifest) vs detected (cleaning log)

This is the key control: every injected defect should be detected by exactly one rule, and every difference must
be explainable.

In [13]:
def logged(rule, table, column=None, issue_contains=None):
    m = (log["rule_id"] == rule) & (log["table"] == table)
    if column:
        m &= log["column"] == column
    if issue_contains:
        m &= log["issue"].str.contains(issue_contains, regex=False)
    return int(log.loc[m, "rows_affected"].sum())

ms_cascade = int((~ms["appointment_id"].isin(ca["appointment_id"])).sum())
rec = [
    ("customers", "duplicate_rows", "DQ01", logged("DQ01", "customers"), ""),
    ("customers", "region_format_variants", "DQ03", logged("DQ03", "customers", "region"), ""),
    ("vehicles", "impossible_model_year", "DQ15", logged("DQ15", "vehicles"), ""),
    ("vehicles", "impossible_mileage", "DQ16", logged("DQ16", "vehicles"), ""),
    ("vehicles", "model_name_variants", "DQ03", logged("DQ03", "vehicles", "model"), ""),
    ("appointments", "status_format_variants", "DQ20", logged("DQ20", "appointments"), ""),
    ("appointments", "ddmmyyyy_dates", "DQ04", logged("DQ04", "appointments"), ""),
    ("appointments", "missing_center_id", "DQ07", logged("DQ07", "appointments", issue_contains="Missing service centre"), ""),
    ("appointments", "orphan_vehicle_id", "DQ06", logged("DQ06", "appointments"), ""),
    ("appointments", "duplicate_rows", "DQ01", logged("DQ01", "appointments"), ""),
    ("work_orders", "start_end_swapped", "DQ08", logged("DQ08", "work_orders"), ""),
    ("work_orders", "missing_technician_id + orphan_technician_id", "DQ10", logged("DQ10", "work_orders"), "one rule covers both defects"),
    ("work_orders", "missing_start_time", "DQ09", logged("DQ09", "work_orders"),
     f"{ms_cascade} row's parent appointment was an orphan removed by DQ06 (cascade DQ18)"),
    ("work_orders", "service_type_format_variants", "DQ03", logged("DQ03", "work_orders"), ""),
    ("work_orders", "duplicate_rows", "DQ01", logged("DQ01", "work_orders"), ""),
    ("financials", "negative_amounts", "DQ11", logged("DQ11", "financials"), "split across revenue and labor_cost"),
    ("financials", "missing_parts_cost", "DQ12", logged("DQ12", "financials"), ""),
    ("financials", "revenue_decimal_shift_x100", "DQ13", logged("DQ13", "financials"), ""),
    ("financials", "duplicate_rows", "DQ01", logged("DQ01", "financials"), ""),
    ("feedback", "rating_out_of_range", "DQ14", logged("DQ14", "feedback", issue_contains="outside"), ""),
    ("feedback", "missing_rating", "DQ14", logged("DQ14", "feedback", issue_contains="Missing"), ""),
    ("feedback", "duplicate_rows", "DQ01", logged("DQ01", "feedback"), ""),
    ("part_usage", "non_positive_quantity", "DQ17", logged("DQ17", "part_usage", "quantity"), ""),
    ("part_usage", "orphan_part_id", "DQ17", logged("DQ17", "part_usage", "part_id"), ""),
]
def injected(table, key):
    return sum(manifest[table][k.strip()] for k in key.split("+"))
recon = pd.DataFrame(rec, columns=["table", "injected_defect", "rule", "detected", "note"])
recon.insert(2, "injected", [injected(t, k) for t, k in zip(recon["table"], recon["injected_defect"])])
recon["status"] = np.where(recon["injected"] == recon["detected"], "Match", "Explained")
print(recon["status"].value_counts().to_dict())
recon

{'Match': 23, 'Explained': 1}


,table,injected_defect,injected,rule,detected,note,status
0,customers,duplicate_rows,13,DQ01,13,,Match
1,customers,region_format_variants,33,DQ03,33,,Match
2,vehicles,impossible_model_year,21,DQ15,21,,Match
3,vehicles,impossible_mileage,15,DQ16,15,,Match
4,vehicles,model_name_variants,26,DQ03,26,,Match
5,appointments,status_format_variants,107,DQ20,107,,Match
6,appointments,ddmmyyyy_dates,214,DQ04,214,,Match
7,appointments,missing_center_id,40,DQ07,40,,Match
8,appointments,orphan_vehicle_id,26,DQ06,26,,Match
9,appointments,duplicate_rows,160,DQ01,160,,Match


**Result:** 23 of 24 defect classes reconcile exactly; the single difference (missing start time: 24 injected,
23 logged) is a work order whose appointment was itself an injected orphan and was correctly removed by the
cascade rule before DQ09 ran.

**Side-effect worth knowing (DQ19):** after DQ17 drops defective part lines, 114 work orders' billed
`parts_cost` no longer equals the sum of their surviving part lines. The billing record is kept as the system
of record and the mismatch is reported, not "fixed".

## 11. Post-cleaning validation

In [14]:
fs = clean["fact_service"]
checks = pd.DataFrame([
    ("Work orders with end_time < start_time", int((fs["end_time"] < fs["start_time"]).sum())),
    ("Negative revenue or cost", int((fs[["revenue", "labor_cost", "parts_cost"]] < 0).any(axis=1).sum())),
    ("Ratings outside 1-5", int((~fs["rating"].dropna().between(1, 5)).sum())),
    ("Work orders without financials", int(fs["revenue"].isna().sum())),
    ("Work orders without centre", int(fs["center_id"].isna().sum())),
    ("Duplicate work_order_id", int(fs["work_order_id"].duplicated().sum())),
    ("Appointments with orphan vehicle", int((~clean["appointments"]["vehicle_id"].isin(clean["vehicles"]["vehicle_id"])).sum())),
    ("Work orders with orphan appointment", int((~clean["work_orders"]["appointment_id"].isin(clean["appointments"]["appointment_id"])).sum())),
    ("Completed appointments without a work order",
     int((clean["appointments"]["status"].eq("Completed") & ~clean["appointments"]["appointment_id"].isin(clean["work_orders"]["appointment_id"])).sum())),
    ("Work orders on non-completed appointments",
     int(clean["work_orders"]["appointment_id"].isin(clean["appointments"].loc[clean["appointments"]["status"] != "Completed", "appointment_id"]).sum())),
], columns=["check", "violations"])
checks

,check,violations
0,Work orders with end_time < start_time,0
1,Negative revenue or cost,0
2,Ratings outside 1-5,0
3,Work orders without financials,0
4,Work orders without centre,0
5,Duplicate work_order_id,0
6,Appointments with orphan vehicle,0
7,Work orders with orphan appointment,0
8,Completed appointments without a work order,0
9,Work orders on non-completed appointments,0


## 12. Derived fields created for analysis (spec §9)

`fact_service` adds the KPI fields required by the specification. A quick sanity view:

In [15]:
derived = ["turnaround_hours", "wait_hours", "late_hours", "on_time_flag", "duration_variance_hours", "parts_delay_flag",
           "total_cost", "profit", "cost_variance", "cost_variance_pct", "cost_overrun_flag", "caused_repeat_visit"]
fs[derived].describe(include="all").T[["count", "mean", "min", "50%", "max"]]

,count,mean,min,50%,max
turnaround_hours,24092.0,11.16565,0.38,2.57,577.37
wait_hours,24069.0,1.19966,0.0,0.32,71.99
late_hours,24092.0,7.316876,0.0,0.0,572.12
on_time_flag,24092,NaN,NaN,NaN,NaN
duration_variance_hours,24092.0,0.310464,-2.56,0.19,4.99
parts_delay_flag,24092,NaN,NaN,NaN,NaN
total_cost,24092.0,2453.617466,88.2,1389.6,51692.4
profit,24092.0,793.878108,-33252.8,703.5,17466.98
cost_variance,24092.0,454.777187,-1152.0,58.8,33342.8
cost_variance_pct,24092.0,0.616819,-0.41,0.0466,103.1178


**Caveats carried into the analysis**

* `wait_hours` and `turnaround_hours` are *clock* hours, so a job checked in late afternoon that waits overnight
  shows a long wait. Medians are reported alongside means.
* `caused_repeat_visit` flags **1,085** jobs followed by a same-type visit within 30 days, while **893** jobs are
  billed as `Rework (No Charge)`. The ~190 difference are coincidental same-type revisits, so the comeback KPI
  slightly over-counts true rework; both measures are shown where it matters.
* 72 work orders without a valid technician (DQ10) are excluded from utilisation, which therefore understates true
  workload by roughly 0.3%.

## Summary
* Every injected defect class is detected; data retention is **≥ 99%** in every table.
* The cleaned, validated tables (`data/cleaned/`) are the single source for SQL, the dashboard and notebook 03.